In [1]:
import random
from pathlib import Path

import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import auc, average_precision_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

In [2]:
SEED = 492
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
g = torch.Generator()
g.manual_seed(SEED)

In [3]:
data_path = Path("../../data/ISIC")
ground_truth_path = data_path / "ISIC_2024_Training_GroundTruth.csv"
metadata_path = data_path / "metadata.csv"
model_path = Path("../../models")

In [4]:
ground_truth_df = pl.read_csv(ground_truth_path)
metadata_df = pl.read_csv(metadata_path)

df = ground_truth_df.join(metadata_df, on="isic_id", how="inner")

patient_stats = df.group_by("patient_id").agg(pl.col("malignant").max().alias("has_malignant")).sort("patient_id")

train_p, val_p = train_test_split(
    patient_stats, test_size=0.1, random_state=SEED, stratify=patient_stats["has_malignant"]
)

train_ids = train_p["patient_id"].to_list()
val_ids = val_p["patient_id"].to_list()

train_df = df.filter(pl.col("patient_id").is_in(train_ids))
val_df = df.filter(pl.col("patient_id").is_in(val_ids))

In [5]:
train_malignant = train_df.filter(pl.col("malignant") == 1)
train_benign = train_df.filter(pl.col("malignant") == 0)

capped_benign = (
    train_benign.sample(fraction=1.0, shuffle=True, seed=SEED)
    .group_by("patient_id")
    .head(20)
    .select(train_df.columns)
)

final_train_df = pl.concat([train_malignant, capped_benign])

print(f"Train Malignant: {train_malignant.height}")
print(f"Train Benign (capped): {capped_benign.height}")
print(f"Total Train: {final_train_df.height}")

Train Malignant: 348
Train Benign (capped): 18392
Total Train: 18740


In [6]:
tab_categorical = ["sex", "anatom_site_general"]
tab_numerical = ["age_approx", "clin_size_long_diam_mm", "tbp_lv_areaMM2", "tbp_lv_eccentricity"]

median_age = final_train_df["age_approx"].median()
mode_sex = final_train_df["sex"].drop_nulls().mode()[0]

final_train_df = final_train_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown"),
)

val_df = val_df.with_columns(
    pl.col("age_approx").fill_null(median_age),
    pl.col("sex").fill_null(mode_sex),
    pl.col("anatom_site_general").fill_null("unknown"),
)

In [7]:
exprs = []
for c in tab_numerical:
    exprs.append(pl.col(c).mean().alias(f"{c}_mean"))
    exprs.append(pl.col(c).std().alias(f"{c}_std"))

num_stats = final_train_df.select(exprs)

for col in tab_numerical:
    mean = num_stats.item(0, f"{col}_mean")
    std = num_stats.item(0, f"{col}_std")
    final_train_df = final_train_df.with_columns(((pl.col(col) - mean) / std).alias(col))
    val_df = val_df.with_columns(((pl.col(col) - mean) / std).alias(col))

In [8]:
tab_features = list(tab_numerical)

for col in tab_categorical:
    categories = final_train_df[col].unique().to_list()
    for cat in categories:
        col_name = f"{col}_{cat}"
        final_train_df = final_train_df.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        val_df = val_df.with_columns((pl.col(col) == cat).cast(pl.Int8).alias(col_name))
        tab_features.append(col_name)

final_train_df = final_train_df.drop(tab_categorical)
val_df = val_df.drop(tab_categorical)

In [9]:
class ISICTabularDataset(Dataset):
    def __init__(self, dataframe: pl.DataFrame, tabular_features: list[str]):
        self.df = dataframe
        self.tab_features = tabular_features

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.row(idx, named=True)
        tab_data = torch.tensor([row[feat] for feat in self.tab_features], dtype=torch.float32)
        label = torch.tensor(row["malignant"], dtype=torch.float32)
        return tab_data, label

In [10]:
train_dataset = ISICTabularDataset(final_train_df, tab_features)
val_dataset = ISICTabularDataset(val_df, tab_features)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [11]:
class TabularModel(nn.Module):
    def __init__(self, tab_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, tab):
        return self.net(tab)

In [12]:
device = torch.device("cuda")
model = TabularModel(tab_dim=len(tab_features)).to(device)

train_malignant_count = final_train_df.filter(pl.col("malignant") == 1).height
train_benign_count = final_train_df.filter(pl.col("malignant") == 0).height

pos_weight = torch.tensor([train_benign_count / train_malignant_count]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=3)
scaler = torch.amp.GradScaler("cuda")

In [13]:
best_ap = 0.0
best_recall_at_spec = 0.0
early_stopping_counter = 0
early_stopping_patience = 10
num_epochs = 50

model_path.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]", leave=False)
    for tabs, labels in train_loop:
        tabs, labels = tabs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(tabs)
            loss = criterion(outputs, labels.unsqueeze(1))
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
        train_loop.set_postfix(loss=loss.item())

    avg_train_loss = train_loss / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_preds = []
    all_labels = []
    val_loop = tqdm(val_loader, desc=f"Epoch {epoch + 1}/{num_epochs} [Val]", leave=False)
    with torch.no_grad():
        for tabs, labels in val_loop:
            tabs, labels = tabs.to(device), labels.to(device)
            with torch.amp.autocast(device_type="cuda"):
                outputs = model(tabs)
                loss = criterion(outputs, labels.unsqueeze(1))
            val_loss += loss.item()
            all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)

    ap = average_precision_score(all_labels, all_preds)
    fpr, tpr, thresholds = roc_curve(all_labels, all_preds)

    valid_indices = np.where(fpr <= 0.10)[0]
    recall_at_spec = tpr[valid_indices[-1]]

    scheduler.step(ap)

    if ap > best_ap:
        best_ap = ap
        early_stopping_counter = 0
    else:
        early_stopping_counter += 1

    if recall_at_spec > best_recall_at_spec:
        best_recall_at_spec = recall_at_spec
        torch.save(model.state_dict(), model_path / "best_tabular_model.pt")

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | "
        f"AP: {ap:.4f} | Recall@90Spec: {recall_at_spec:.4f}"
    )

    if early_stopping_counter >= early_stopping_patience:
        print(f"Early stopping triggered at epoch {epoch + 1}")
        break

Epoch 1/50 | Train Loss: 1.3455 | Val Loss: 0.7368 | AP: 0.0084 | Recall@90Spec: 0.4667


Epoch 2/50 | Train Loss: 1.3315 | Val Loss: 0.7310 | AP: 0.0083 | Recall@90Spec: 0.4667


Epoch 3/50 | Train Loss: 1.3290 | Val Loss: 0.7193 | AP: 0.0086 | Recall@90Spec: 0.4444


Epoch 4/50 | Train Loss: 1.2776 | Val Loss: 0.7095 | AP: 0.0088 | Recall@90Spec: 0.4444


Epoch 5/50 | Train Loss: 1.2357 | Val Loss: 0.6969 | AP: 0.0091 | Recall@90Spec: 0.4222


...

Epoch 46/50 | Train Loss: 0.7815 | Val Loss: 0.4618 | AP: 0.0274 | Recall@90Spec: 0.6000


Epoch 47/50 | Train Loss: 0.7921 | Val Loss: 0.4444 | AP: 0.0294 | Recall@90Spec: 0.6000


Epoch 48/50 | Train Loss: 0.7757 | Val Loss: 0.4652 | AP: 0.0281 | Recall@90Spec: 0.6000


Epoch 49/50 | Train Loss: 0.7625 | Val Loss: 0.4415 | AP: 0.0297 | Recall@90Spec: 0.6000


Epoch 50/50 | Train Loss: 0.7526 | Val Loss: 0.4485 | AP: 0.0315 | Recall@90Spec: 0.6000


In [14]:
model.load_state_dict(torch.load(model_path / "best_tabular_model.pt", weights_only=True))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for tabs, labels in tqdm(val_loader, desc="Evaluating Best Model"):
        tabs, labels = tabs.to(device), labels.to(device)
        with torch.amp.autocast(device_type="cuda"):
            outputs = model(tabs)
        all_preds.extend(torch.sigmoid(outputs).squeeze(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

auc_roc = roc_auc_score(all_labels, all_preds)
ap = average_precision_score(all_labels, all_preds)

fpr, tpr, thresholds = roc_curve(all_labels, all_preds)
valid_idx_90 = np.where(fpr <= 0.10)[0]
threshold_90_spec = thresholds[valid_idx_90[-1]]
recall_90_spec = tpr[valid_idx_90[-1]]

valid_idx_95 = np.where(fpr <= 0.05)[0]
threshold_95_spec = thresholds[valid_idx_95[-1]]
recall_95_spec = tpr[valid_idx_95[-1]]

preds_binary = (all_preds >= threshold_90_spec).astype(int)
tn, fp, fn, tp = confusion_matrix(all_labels, preds_binary).ravel()

print(f"AUC-ROC: {auc_roc:.4f}")
print(f"Average Precision: {ap:.4f}")
print("--- @ 90% Spec ---")
print(f"Threshold: {threshold_90_spec:.4f} | Recall: {recall_90_spec:.4f}")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")
print("--- @ 95% Spec ---")
print(f"Threshold: {threshold_95_spec:.4f} | Recall: {recall_95_spec:.4f}")

Evaluating Best Model: 100%|██████████| 47/47 [00:00<00:00, 67.38it/s]


AUC-ROC: 0.8622
Average Precision: 0.0133
--- @ 90% Spec ---
Threshold: 0.6997 | Recall: 0.6444
TP: 29 | FP: 4798 | TN: 43190 | FN: 16
--- @ 95% Spec ---
Threshold: 0.8110 | Recall: 0.4667


In [15]:
def p_auc_tpr(v_gt, v_pred, min_tpr=0.80):
    v_gt_flipped = abs(np.asarray(v_gt) - 1)
    v_pred_flipped = abs(np.asarray(v_pred) - 1)
    max_fpr = abs(1 - min_tpr)

    fpr, tpr, _ = roc_curve(v_gt_flipped, v_pred_flipped)

    stop = np.searchsorted(fpr, max_fpr, "right")
    x_interp = [fpr[stop - 1], fpr[stop]]
    y_interp = [tpr[stop - 1], tpr[stop]]

    tpr_adj = np.append(tpr[:stop], np.interp(max_fpr, x_interp, y_interp))
    fpr_adj = np.append(fpr[:stop], max_fpr)

    return auc(fpr_adj, tpr_adj)


isic_pauc = p_auc_tpr(all_labels, all_preds, min_tpr=0.80)
print(f"ISIC 2024 Official Metric (pAUC > 80% TPR): {isic_pauc:.5f}")

ISIC 2024 Official Metric (pAUC > 80% TPR): 0.10782
